# Trabalho IA 2025/2026 - PopOut, MCTS e Decision Trees ID3

Este notebook apresenta uma solução para o jogo **PopOut**, uma variante do Connect Four onde cada jogador pode colocar peças no topo de uma coluna ou retirar uma peça própria da base("pop").

Além da implementação do jogo, o notebook inclui:

- uma IA baseada em **Monte Carlo Tree Search**;
- geração de datasets a partir de jogadas escolhidas pelo MCTS;
- uma árvore de decisão **ID3** implementada de raiz, sem `scikit-learn`;
- exemplo da implementação da árvore de decisão no dataset `iris`;
- um jogador baseado na árvore ID3;
- avaliação e comparação das AIs criadas.


## Imports e estruturas auxiliares

Esta célula importa bibliotecas standard usadas em todo o notebook:

- `random`, `math` e `copy` para o MCTS e para manipular estados do jogo;
- `csv`, `time` e `datetime` para gerar datasets e acompanhar o progresso;
- `Counter` e `defaultdict` para contagens usadas no ID3 e na deteção de repetições;
- `dataclass` e tipos de `typing` para tornar a árvore ID3 mais legível.


In [ ]:
import random
import math
import copy
import csv
import time
import datetime
import numpy as np
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Any, Dict, Optional

## Classe `Poupout`: representação e regras do jogo

A classe `Poupout` é o núcleo do projeto. Ela representa o estado atual do jogo e disponibiliza os métodos necessários para jogar e avaliar posições.

Principais responsabilidades:

- criar e guardar o tabuleiro;
- executar jogadas `put`, `pop` e `draw`;
- alternar o jogador atual;
- verificar vitórias em linhas, colunas e diagonais;
- detetar tabuleiro cheio e repetição de estados;
- produzir cópias do estado para o MCTS;
- devolver a lista de jogadas legais para o jogador atual.
- criação e avaliação de tabuleiros aleatórios para a criação dos datasets

In [3]:
class Poupout:
    def __init__(self,rows,cols,moved="O",to_move="X"):
        self.rows=rows
        self.cols=cols
        self.moved=moved
        self.to_move=to_move
        self.board=list()
        for c in range(cols):
            line=list()
            for r in range(rows):
                line.append("-")
            self.board.append(line)
        self.n_pieces=0
        self.repeated=False
        self.last_move=None
        self.draw=False
    
    def display(self):
        print("  " + " ".join(str(i) for i in range(self.cols)))
        for r in range(self.rows):
            print("  ", end="")
            for c in range(self.cols):
                print(self.board[c][r], end="")
            print("")

    def put(self, column):
        if self.board[column][0]!="-":
            raise Exception("Ação impossível")
        i=self.rows-1
        while True:
            if i<0:
                return
            if self.board[column][i]=="-":
                self.board[column][i]=self.to_move
                self.n_pieces+=1
                return
            else:
                i-=1
    
    def pop(self, column):
        if self.board[column][-1]!=self.to_move:
            raise Exception("Ação impossível")
        self.board[column]=["-"] + self.board[column][:-1]
        self.n_pieces-=1

    def change_to_move(self):
        temp=self.to_move
        self.to_move=self.moved
        self.moved=temp

    def make_move(self, move, column):
        try:
            if move=="put":
                self.put(column)
            elif move=="pop":
                self.pop(column)
            elif move=="draw" and (self.repeated or self.check_full()):
                self.draw=True
            else:
                return False
            self.last_move=(move,column)
            return True
        except:
            return False
        
    def check_row(self, row):
        line=""
        for i in range(self.cols):
            line+=self.board[i][row]
        moved=self.moved*4
        if line.find(moved)!=-1:
            return self.moved
        to_move=self.to_move*4
        if line.find(to_move)!=-1:
            return self.to_move
        return None
    
    def check_col(self, col):
        line=""
        for i in range(self.rows):
            line+=self.board[col][i]
        moved=self.moved*4
        if line.find(moved)!=-1:
            return self.moved
        to_move=self.to_move*4
        if line.find(to_move)!=-1:
            return self.to_move
        return None

    def check_diag1(self,row,col):
        cells=min(self.rows-row,self.cols-col)
        if cells<4:
            return None
        line=""
        for i in range(cells):
            line+=self.board[col+i][row+i]
        moved=self.moved*4
        if line.find(moved)!=-1:
            return self.moved
        to_move=self.to_move*4
        if line.find(to_move)!=-1:
            return self.to_move
        return None
    
    def check_diag2(self,row,col):
        cells=min(self.rows-row,col+1)
        if cells<4:
            return None
        line=""
        for i in range(cells):
            line+=self.board[col-i][row+i]
        moved=self.moved*4
        if line.find(moved)!=-1:
            return self.moved
        to_move=self.to_move*4
        if line.find(to_move)!=-1:
            return self.to_move
        return None
        
    def check_win(self):
        if self.last_move is None:
            return None
        move, column= self.last_move
        adversary_win=False
        if move=="put":
            row=self.board[column].index(self.moved)
            winner=self.check_row(row)
            if winner is not None:
                return winner
            winner=self.check_col(column)
            if winner is not None:
                return winner    
            winner=self.check_diag1(max(0,row-column),max(0,column-row))
            if winner is not None:
                return winner
            winner=self.check_diag2(max(0,row-(self.cols-1-column)),min(self.cols-1,row+column))
            if winner is not None:
                return winner
        if move=="pop":
            for row in range(self.rows-1, -1, -1):
                if self.board[column][row]=="-":
                    break
                winner=self.check_row(row)
                if winner == self.moved:
                    return winner
                elif winner==self.to_move:
                    adversary_win=True
                winner=self.check_diag1(max(0,row-column),max(0,column-row))
                if winner == self.moved:
                    return winner
                elif winner==self.to_move:
                    adversary_win=True
                winner=self.check_diag2(max(0,row-(self.cols-1-column)),min(self.cols-1,row+column))
                if winner == self.moved:
                    return winner
                elif winner==self.to_move:
                    adversary_win=True
        if adversary_win:
            return self.to_move
        return None
    
    def check_full(self):
        return self.n_pieces==self.rows*self.cols

    def check_repeat(self, states, states_dict):
        if self.repeated:
            return True
        elif states_dict[self.n_pieces]>=3:
            for s in states[self.n_pieces]:
                if states[self.n_pieces].count(s)>=3:
                    self.repeated=True
                    return True
        return False
    
    def clone(self):
        new = Poupout(self.rows, self.cols, self.moved, self.to_move)
        new.board = copy.deepcopy(self.board)
        new.n_pieces = self.n_pieces
        new.repeated = self.repeated
        new.last_move = self.last_move
        # não copiamos states/states_dict para poupar memória nas simulações
        return new

    def legal_moves(self):
        moves = []
        for c in range(self.cols):
            if self.board[c][0] == "-":
                moves.append(("put", c))
            if self.board[c][-1] == self.to_move:
                moves.append(("pop", c))
        if self.repeated or self.check_full():
            moves.append(("draw",0))
        return moves

    def is_terminal(self):
        return (self.check_win() is not None
                or self.draw)

    def get_result(self, maximizing_player):
        winner = self.check_win()
        if winner == maximizing_player:
            return 1
        if winner is not None:
            return -1
        else:
            return 0

## Classes `MCTSNode` e `MCTS`: jogador por Monte Carlo Tree Search

O MCTS constrói uma árvore de possibilidades a partir do estado atual. Cada nó guarda:

- o estado do jogo naquele ponto;
- a jogada que levou até esse estado;
- o nó pai e os filhos;
- número de visitas;
- pontuação acumulada;
- jogadas ainda não exploradas.

Cada iteração do MCTS passa por quatro fases:

1. **Selection**: desce pela árvore usando UCT.
2. **Expansion**: cria um novo filho com uma jogada ainda não testada.
3. **Simulation**: joga aleatoriamente até ao fim ou até uma profundidade máxima.
4. **Backpropagation**: propaga o resultado da simulação até à raiz.

No final, a jogada escolhida é a do filho da raiz com mais visitas, porque tende a ser a opção mais estável.


In [4]:
class MCTSNode:
    def __init__(self, game_state, parent=None, move=None):
        self.state = game_state
        self.parent = parent
        self.move = move
        self.children = []
        self.wins = 0.0
        self.visits = 0
        self._untried = game_state.legal_moves()
        random.shuffle(self._untried)

    def is_fully_expanded(self):
        return len(self._untried) == 0

    def uct_score(self, c=math.sqrt(2)):
        if self.visits == 0:
            return float("inf")
        return self.wins / self.visits + c * math.sqrt(math.log(self.parent.visits) / self.visits)

    def best_child(self, c=math.sqrt(2)):
        return max(self.children, key=lambda n: n.uct_score(c))

class MCTS:
    def __init__(self, iterations=600, c=math.sqrt(2), max_depth=40):
        self.iterations = iterations
        self.c = c
        self.max_depth = max_depth

    def choose_move(self, game_state):
        root = MCTSNode(game_state.clone())
        if not root._untried:
            return None
        original_player = game_state.to_move
        for _ in range(self.iterations):
            node = self._select(root)
            if not node.state.is_terminal():
                node = self._expand(node)
            result = self._simulate(node, original_player)
            self._backpropagate(node, result, original_player)
        return max(root.children, key=lambda n: n.visits).move

    def _select(self, node):
        while not node.state.is_terminal() and node.is_fully_expanded() and node.children:
            node = node.best_child(self.c)
        return node

    def _expand(self, node):
        if node._untried:
            move = node._untried.pop()
            new_game = node.state.clone()
            new_game.make_move(move[0], move[1])
            new_game.change_to_move()
            child = MCTSNode(new_game, parent=node, move=move)
            node.children.append(child)
            return child
        return node

    def _rollout_move(self, sim):
        moves = sim.legal_moves()
        return random.choice(moves)

    def _simulate(self, node, original_player):
        sim = node.state.clone()
        depth = 0
        while not sim.is_terminal() and depth < self.max_depth:
            moves = sim.legal_moves()
            if not moves:
                break
            action, col = self._rollout_move(sim)
            sim.make_move(action, col)
            sim.change_to_move()
            depth += 1
        return sim.get_result(original_player)

    def _backpropagate(self, node, result, original_player):
        while node is not None:
            node.visits += 1
            node_player = node.state.moved
            if node_player == original_player:
                node.wins += result
            else:
                node.wins -= result
            node = node.parent


## Função `play`: execução de uma partida IA vs IA

A função `play(player_X, player_O)` permite observar uma partida completa entre dois jogadores automáticos.


In [79]:
def play(player_X, player_O, show=True):
    ai_x = player_X
    ai_o = player_O

    jogo = Poupout(6, 7)
    state_counts = defaultdict(int)
    state_counts[jogo.board_key()] += 1

    while True:
        if show:
            jogo.display()
        winner = jogo.check_win()
        if winner is not None:
            if show:
                print(f"Parabens, {winner} ganhou!")
            break
        if jogo.draw:
            if show:
                print("Empate!")
            break

        jogador = ai_x if jogo.to_move == "X" else ai_o
        if show:
            print(f"IA ({jogo.to_move}) a pensar...")
        jogada = jogador.choose_move(jogo)
        action, col = jogada
        jogo.make_move(action, col)
        if show:
            print(f"  -> IA jogou: {action} coluna {col}")
        if jogada not in jogo.legal_moves():
            if show:
                print(f"Parabens, {jogo.moved} ganhou!")
            return jogo.moved
        state_counts[jogo.board_key()] += 1
        jogo.check_repeat(state_counts)
        jogo.change_to_move()
    if winner is not None:
        return winner
    return None

## Exemplo de partida entre dois MCTS

A célula seguinte cria dois jogadores MCTS e coloca-os a jogar. Como valores grandes de iterações podem demorar, podes reduzir `Iter_Jogador_X` e `Iter_Jogador_O` quando estiveres apenas a testar o notebook.


In [6]:
Iter_Jogador_X=10000
Iter_Jogador_O=10000
Jogador_X=MCTS(iterations=Iter_Jogador_X)
Jogador_O=MCTS(iterations=Iter_Jogador_O)
play(Jogador_X, Jogador_O)

  0 1 2 3 4 5 6
  -------
  -------
  -------
  -------
  -------
  -------
IA (X) a pensar...
  -> IA jogou: put coluna 3
  0 1 2 3 4 5 6
  -------
  -------
  -------
  -------
  -------
  ---X---
IA (O) a pensar...
  -> IA jogou: put coluna 3
  0 1 2 3 4 5 6
  -------
  -------
  -------
  -------
  ---O---
  ---X---
IA (X) a pensar...
  -> IA jogou: put coluna 2
  0 1 2 3 4 5 6
  -------
  -------
  -------
  -------
  ---O---
  --XX---
IA (O) a pensar...
  -> IA jogou: put coluna 4
  0 1 2 3 4 5 6
  -------
  -------
  -------
  -------
  ---O---
  --XXO--
IA (X) a pensar...
  -> IA jogou: put coluna 1
  0 1 2 3 4 5 6
  -------
  -------
  -------
  -------
  ---O---
  -XXXO--
IA (O) a pensar...
  -> IA jogou: put coluna 0
  0 1 2 3 4 5 6
  -------
  -------
  -------
  -------
  ---O---
  OXXXO--
IA (X) a pensar...
  -> IA jogou: put coluna 0
  0 1 2 3 4 5 6
  -------
  -------
  -------
  -------
  X--O---
  OXXXO--
IA (O) a pensar...
  -> IA jogou: put coluna 2
  0 1 2 3 4 5 6


'O'

## Geração do dataset PopOut

Para treinar a árvore ID3, cada posição do jogo é transformada num conjunto de atributos.

A função `state_to_features` cria uma linha do dataset com:

- uma coluna para cada casa do tabuleiro, como `c0_r0`, `c0_r1`, ...;
- o jogador que tem de jogar (`to_move`).
- os movimentos permitidos nesse estado, como `put_0`, `put_1`, ...;

A função `generate_popout_dataset` joga várias partidas usando MCTS e guarda, para cada estado, a jogada escolhida pelo MCTS. Essa jogada fica na coluna `move`, por exemplo `put_3` ou `pop_2`.


In [ ]:
def state_to_features(game):
    row = {}
    for c in range(game.cols):
        for r in range(game.rows):
            row[f"c{c}_r{r}"] = game.board[c][r]
    row["to_move"] = game.to_move
    for m in ["put", "pop"]:
        for c in range(game.cols):
            if (m,c) in game.legal_moves():
                row[f"{m}_{c}"]="T"
            else:
                row[f"{m}_{c}"]="F"
    return row

def generate_popout_dataset(n_base_games=2, n_random_games=0, mcts_iterations=120, max_depth=40, min_moves=3, outfile="popout_dataset.csv", show_progress=False):
    start=time.time()
    ai = MCTS(iterations=mcts_iterations, max_depth=max_depth)
    game_sizes=[]
    game_sizes.extend([0]*n_base_games)
    game_sizes=np.random.normal(loc=15, scale=5, size=n_random_games)
    game_sizes=list(game_sizes)
    for g in range(len(game_sizes)):
        game_sizes[g]=int(game_sizes[g])
        if game_sizes[g]<0:
            game_sizes[g]=0
        elif game_sizes[g]>42:
            game_sizes[g]=42
    for g in range(43):
        game_sizes.extend([g]*n_base_games)
    random.shuffle(game_sizes)
    rows = []
    perc=0

    for i,g in enumerate(game_sizes):
        if show_progress and (i/len(game_sizes))*100>=perc+10:
            perc+=10
            cur=time.time()
            elapsed=cur-start
            expected=(elapsed/perc)*100
            left=expected-elapsed
            print(f"{perc}%  Faltam {str(datetime.timedelta(seconds=left))}")
        while True:  
            new_rows=[]  
            game = Poupout(6, 7)
            game.random_board(g)
            if game.full_check_win() is not None:
                continue
            state_counts = defaultdict(int)
            state_counts[game.board_key()] += 1

            n_moves=0
            while True:
                if game.is_terminal():
                    break
                move = ai.choose_move(game)
                features = state_to_features(game)
                features["move"] = f"{move[0]}_{move[1]}"
                new_rows.append(features)
                game.make_move(move[0], move[1])
                n_moves+=1
                game.check_repeat(state_counts)
                game.change_to_move()
            if n_moves<min_moves:
                continue
            rows.extend(new_rows)
            break   

        fieldnames = list(rows[0].keys())
        with open(outfile, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)
    if show_progress:
        print("Done")

## Geração do dataset PopOut

Para treinar a árvore ID3, cada posição do jogo é transformada num conjunto de atributos.

A função `state_to_features` cria uma linha do dataset com:

- uma coluna para cada casa do tabuleiro, como `c0_r0`, `c0_r1`, ...;
- o jogador que tem de jogar (`to_move`).
- os movimentos permitidos nesse estado, como `put_0`, `put_1`, ...;

A função `generate_popout_dataset` joga várias partidas usando MCTS e guarda, para cada estado, a jogada escolhida pelo MCTS. Essa jogada fica na coluna `move`, por exemplo `put_3` ou `pop_2`.


In [ ]:
generate_popout_dataset(n_base_games=3, n_random_games=100, mcts_iterations=100, min_moves=3, outfile="teste.csv", show_progress=True)

10%  Faltam 0:00:28.340203
20%  Faltam 0:00:30.563012
30%  Faltam 0:00:25.503120
40%  Faltam 0:00:21.859458
50%  Faltam 0:00:59.159265
60%  Faltam 0:01:01.324748
70%  Faltam 0:00:52.241553
80%  Faltam 0:00:32.606548
90%  Faltam 0:00:14.915685
Done


Criação de datasets base com jogos a partir do começo. É possível observar que, com o aumento de iterações para o MTCS utilizado, o número de jogos simulados tem de diminuir para que a geração dos datasets não demore demasiado.

In [ ]:
print("Dataset de 100 iterações:")
generate_popout_dataset(n_base_games=500, mcts_iterations=100, outfile="popout_dataset_100_default.csv", show_progress=True)
print()
print("Dataset de 1000 iterações:")
generate_popout_dataset(n_base_games=200, mcts_iterations=1000, outfile="popout_dataset_100_default.csv", show_progress=True)
print()
print("Dataset de 10000 iterações:")
generate_popout_dataset(n_base_games=100, mcts_iterations=10000, outfile="popout_dataset_100_default.csv", show_progress=True)

Criação de datasets com random_boards. Desta forma podemos evitar repetir constantemente os mesmo tabuleiros, garantindo situações diversas no dataset assim como situações relevantes com a verificação de movimentos mínimos por cada estado.

In [ ]:
print("Dataset random de 100 iterações:")
generate_popout_dataset(n_base_games=5, n_random_games=500, mcts_iterations=100, option="random_board", outfile="popout_dataset_100_random.csv", show_progress=True)
print()
print("Dataset random de 1000 iterações:")
generate_popout_dataset(n_base_games=5, n_random_games=200, mcts_iterations=1000, option="random_board", outfile="popout_dataset_1000_random.csv", show_progress=True)
print()
print("Dataset random de 10000 iterações:")
generate_popout_dataset(n_base_games=5, n_random_games=100, mcts_iterations=10000, option="random_board", outfile="popout_dataset_10000_random.csv", show_progress=True)

Dataset de 100 iterações:
10%  Faltam 0:19:26.206769
20%  Faltam 0:11:00.333366
30%  Faltam 0:10:49.875927
40%  Faltam 0:08:12.303174
50%  Faltam 0:07:04.168179
60%  Faltam 0:05:19.568583
70%  Faltam 0:03:48.479259
80%  Faltam 0:02:58.482706
90%  Faltam 0:01:23.857316
Done

Dataset de 1000 iterações:
10%  Faltam 0:36:52.342584
20%  Faltam 0:35:56.195876
30%  Faltam 0:26:39.554133
40%  Faltam 0:21:51.902789
50%  Faltam 0:17:36.978964
60%  Faltam 0:13:19.282301
70%  Faltam 0:09:57.931529
80%  Faltam 0:06:24.615658
90%  Faltam 0:03:09.785529
Done

Dataset de 10000 iterações:
10%  Faltam 3:01:54.703418
20%  Faltam 2:37:33.771412
30%  Faltam 2:08:42.298383
40%  Faltam 1:47:54.564227
50%  Faltam 1:37:24.968971
60%  Faltam 1:16:34.535921
70%  Faltam 0:56:12.656723
80%  Faltam 0:35:11.615266
90%  Faltam 0:18:15.357951
Done


## Árvore de decisão ID3

Esta secção implementa o algoritmo **ID3** sem usar bibliotecas externas de machine learning.

A árvore escolhe, em cada nó, o atributo com maior **ganho de informação**. Para isso usa:

- **entropia**, que mede a impureza das classes;
- **information gain**, que mede quanto um atributo reduz essa impureza;



In [8]:
@dataclass
class ID3Node:
    prediction: Any
    attribute: Optional[str] = None
    children: Optional[Dict[Any, "ID3Node"]] = None
    is_leaf: bool = False

    def classify(self, example):
        if self.is_leaf or self.attribute is None:
            return self.prediction
        value = example.get(self.attribute)
        if self.children and value in self.children:
            return self.children[value].classify(example)
        return self.prediction

    def pretty(self, indent=""):
        if self.is_leaf:
            return indent + f"Classe: {self.prediction}\n"
        s = indent + f"Se {self.attribute}:\n"
        for value, child in sorted(self.children.items(), key=lambda x: str(x[0])):
            s += indent + f"  = {value} ->\n"
            s += child.pretty(indent + "    ")
        return s

class ID3DecisionTree:
    def __init__(self, max_depth=None, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None
        self.label_col = None

    @staticmethod
    def entropy(labels):
        total = len(labels)
        counts = Counter(labels)
        return -sum((cnt/total) * math.log2(cnt/total) for cnt in counts.values() if cnt)

    def information_gain(self, rows, attribute, label_col):
        base = self.entropy([r[label_col] for r in rows])
        total = len(rows)
        parts = defaultdict(list)
        for r in rows:
            parts[r[attribute]].append(r)
        remainder = sum((len(part)/total) * self.entropy([r[label_col] for r in part]) for part in parts.values())
        return base - remainder

    def majority_label(self, rows, label_col):
        return Counter(r[label_col] for r in rows).most_common(1)[0][0]

    def fit(self, rows, label_col="label", attributes=None):
        if not rows:
            raise ValueError("Dataset vazio")
        self.label_col = label_col
        if attributes is None:
            attributes = [a for a in rows[0].keys() if a != label_col]
        self.root = self._id3(rows, attributes, label_col, depth=0)
        return self

    def _id3(self, rows, attributes, label_col, depth):
        labels = [r[label_col] for r in rows]
        majority = self.majority_label(rows, label_col)
        if len(set(labels)) == 1:
            return ID3Node(prediction=labels[0], is_leaf=True)
        if not attributes or len(rows) < self.min_samples_split or (self.max_depth is not None and depth >= self.max_depth):
            return ID3Node(prediction=majority, is_leaf=True)

        gains = [(self.information_gain(rows, a, label_col), a) for a in attributes]
        best_gain, best_attr = max(gains, key=lambda x: x[0])
        if best_gain <= 1e-12:
            return ID3Node(prediction=majority, is_leaf=True)

        node = ID3Node(prediction=majority, attribute=best_attr, children={}, is_leaf=False)
        values = sorted(set(r[best_attr] for r in rows), key=str)
        remaining = [a for a in attributes if a != best_attr]
        for v in values:
            subset = [r for r in rows if r[best_attr] == v]
            node.children[v] = self._id3(subset, remaining, label_col, depth + 1)
        return node

    def predict_one(self, example):
        if self.root is None:
            raise ValueError("A arvore ainda nao foi treinada")
        return self.root.classify(example)

    def predict(self, rows):
        return [self.predict_one(r) for r in rows]

    def score(self, rows, label_col=None):
        label_col = label_col or self.label_col
        preds = self.predict(rows)
        return sum(p == r[label_col] for p, r in zip(preds, rows)) / len(rows)

    def pretty(self):
        return self.root.pretty() if self.root else "<arvore vazia>"

## Funções auxiliares para treino e avaliação

Estas funções são usadas para preparar datasets e avaliar modelos:

- `train_test_split`: divide os dados em treino e teste;
- `discretize_numeric_dataset`: transforma atributos numéricos em bins, necessário para o ID3;
- `load_csv_dataset`: carrega datasets CSV;


In [9]:
def train_test_split(rows, test_ratio=0.25, seed=0):
    random.Random(seed).shuffle(rows)
    cut = int(len(rows) * (1 - test_ratio))
    return rows[:cut], rows[cut:]

def discretize_numeric_dataset(rows, label_col="label", bins=3):
    # Discretizacao por quantis: reduz tamanho da arvore e funciona bem no Iris.
    attrs = [a for a in rows[0] if a != label_col]
    thresholds = {}
    for a in attrs:
        vals = sorted(float(r[a]) for r in rows)
        qs = []
        for b in range(1, bins):
            idx = int(len(vals) * b / bins)
            qs.append(vals[idx])
        thresholds[a] = qs

    new_rows = []
    for r in rows:
        nr = {label_col: r[label_col]}
        for a in attrs:
            x = float(r[a])
            k = 0
            while k < len(thresholds[a]) and x > thresholds[a][k]:
                k += 1
            nr[a] = f"bin{k}"
        new_rows.append(nr)
    return new_rows, thresholds

def load_csv_dataset(path):
    with open(path) as file:
        ds=csv.DictReader(file)
        rows=[]
        for r in ds:
            rows.append(r)
        return rows

## Jogador baseado em ID3

A classe `ID3DecisionTreePlayer` transforma uma árvore ID3 treinada num jogador de PopOut.

Quando recebe um estado do jogo:

1. converte o estado em atributos com `state_to_features`;
2. usa a árvore para prever a jogada;
3. converte a previsão, por exemplo `put_3`, para o formato usado pelo jogo: `("put", 3)`.

Esta IA é mais rápida a decidir do que o MCTS, mas depende da qualidade do dataset usado no treino.


In [10]:
class ID3DecisionTreePlayer:
    def __init__(self, tree):
        self.tree=tree

    def choose_move(self, state):
        game=state_to_features(state)
        move=self.tree.predict_one(game)
        move=move.split("_")
        return (move[0], int(move[1]))

Exemplo: treinar ID3 com dataset PopOut


In [14]:
dataset_1000=load_csv_dataset(path="popout_dataset_1000_random.csv")
tree = ID3DecisionTree(max_depth=10)
tree.fit(dataset_1000, label_col="move")
Player=ID3DecisionTreePlayer(tree)
jogo=Poupout(6,7)
jogo.display()
print(Player.choose_move(jogo))

  0 1 2 3 4 5 6
  -------
  -------
  -------
  -------
  -------
  -------
('put', 3)


In [70]:
dataset_100=load_csv_dataset(path="popout_dataset_100_random.csv")
dataset_1000=load_csv_dataset(path="popout_dataset_1000_random.csv")
dataset_10000=load_csv_dataset(path="popout_dataset_10000_random.csv")
tree1 = ID3DecisionTree(max_depth=20)
tree1.fit(dataset_100, label_col="move")
tree2 = ID3DecisionTree(max_depth=20)
tree2.fit(dataset_10000, label_col="move")
Player_X=ID3DecisionTreePlayer(tree2)
Player_O=ID3DecisionTreePlayer(tree1)
play(Player_X, Player_O)

  0 1 2 3 4 5 6
  -------
  -------
  -------
  -------
  -------
  -------
IA (X) a pensar...
  -> IA jogou: put coluna 3
  0 1 2 3 4 5 6
  -------
  -------
  -------
  -------
  -------
  ---X---
IA (O) a pensar...
  -> IA jogou: put coluna 3
  0 1 2 3 4 5 6
  -------
  -------
  -------
  -------
  ---O---
  ---X---
IA (X) a pensar...
  -> IA jogou: put coluna 4
  0 1 2 3 4 5 6
  -------
  -------
  -------
  -------
  ---O---
  ---XX--
IA (O) a pensar...
  -> IA jogou: put coluna 2
  0 1 2 3 4 5 6
  -------
  -------
  -------
  -------
  ---O---
  --OXX--
IA (X) a pensar...
  -> IA jogou: put coluna 5
  0 1 2 3 4 5 6
  -------
  -------
  -------
  -------
  ---O---
  --OXXX-
IA (O) a pensar...
  -> IA jogou: put coluna 1
  0 1 2 3 4 5 6
  -------
  -------
  -------
  -------
  ---O---
  -OOXXX-
IA (X) a pensar...
  -> IA jogou: put coluna 2
  0 1 2 3 4 5 6
  -------
  -------
  -------
  -------
  --XO---
  -OOXXX-
IA (O) a pensar...
  -> IA jogou: put coluna 0
  0 1 2 3 4 5 6


'O'

## Iris: carregar, discretizar e treinar

Coloca o ficheiro `iris.csv` na mesma pasta do notebook com as colunas: `sepal_length`, `sepal_width`, `petal_length`, `petal_width`, `label`.

In [257]:
# Exemplo para quando tiveres o ficheiro iris.csv
iris_rows = load_csv_dataset("iris.csv")
iris_disc, thresholds = discretize_numeric_dataset(iris_rows, label_col="Species", bins=3)
train, test = train_test_split(iris_disc, test_ratio=0.25, seed=2)
iris_tree = ID3DecisionTree(max_depth=None).fit(train, label_col="Species")
print("Accuracy treino:", iris_tree.score(train, "Species"))
print("Accuracy teste:", iris_tree.score(test, "Species"))
print("Thresholds usados:", thresholds)
print(iris_tree.pretty())

Accuracy treino: 1.0
Accuracy teste: 0.9736842105263158
Thresholds usados: {'Id': [51.0, 101.0], 'SepalLengthCm': [5.4, 6.3], 'SepalWidthCm': [2.9, 3.2], 'PetalLengthCm': [3.0, 4.9], 'PetalWidthCm': [1.0, 1.6]}
Se Id:
  = bin0 ->
    Se SepalLengthCm:
      = bin0 ->
        Classe: Iris-setosa
      = bin1 ->
        Classe: Iris-setosa
      = bin2 ->
        Classe: Iris-versicolor
  = bin1 ->
    Classe: Iris-versicolor
  = bin2 ->
    Classe: Iris-virginica



## Experiência MCTS: número de iterações

Compara jogadores MCTS com diferentes orçamentos de iterações.

In [ ]:
class RandomPlayer():
    def choose_move(self, state):
        moves=state.legal_moves()
        return random.choice(moves)

('put', 6)


In [49]:
def simulate_games(Player1, Player2, n_games):
    games_set1=[]
    games_set2=[]
    p1_wins=0
    p2_wins=0
    for _ in range(n_games//2):
        players={"X":Player1, "O":Player2}
        games_set1.append(play(players["X"], players["O"], show=False))
    p1_wins+=games_set1.count("X")
    p2_wins+=games_set1.count("O")
    for _ in range(n_games//2):
        players={"X":Player2, "O":Player1}
        games_set2.append(play(players["X"], players["O"], show=False))
    p2_wins+=games_set2.count("X")
    p1_wins+=games_set2.count("O")
    games=games_set1 + games_set2
    return (p1_wins, p2_wins, games)

Exemplo da simulação de jogos entre duas AIs:

In [61]:
p1=RandomPlayer()
p2=MCTS(iterations=100)
games=simulate_games(p1,p2,10)
print(games[0])
print(games[1])
print(games[2])

0
10
['O', 'O', 'O', 'O', 'O', 'X', 'X', 'X', 'X', 'X']


Comparação entre AIs MCTS:

In [ ]:
players={"random":RandomPlayer(), "mcts_100":MCTS(iterations=100), "mcts_1000":MCTS(iterations=1000), "mcts_10000":MCTS(iterations=10000)}
player_names=players.keys()
n_games=10
results={}
for p in player_names:
    results[f"{p} wins"]=0
for i,p1 in enumerate(player_names):
    for p2 in player_names[i+1:]:
        game_name=f"{p1}_vs_{p2}"
        result=simulate_games(players[p1],players[p2],n_games)
        results[f"{p1} wins"]+=result[0]
        results[f"{p2} wins"]+=result[1]
        print(f"{game_name} {result}")
for p in results:
    print(f"{p}: {results[p]}")


random_vs_mcts_100 (0, 10, ['O', 'O', 'O', 'O', 'O', 'X', 'X', 'X', 'X', 'X'])
random_vs_mcts_1000 (0, 10, ['O', 'O', 'O', 'O', 'O', 'X', 'X', 'X', 'X', 'X'])
random_vs_mcts_10000 (0, 10, ['O', 'O', 'O', 'O', 'O', 'X', 'X', 'X', 'X', 'X'])
mcts_100_vs_mcts_1000 (2, 8, ['O', 'O', 'O', 'X', 'O', 'X', 'X', 'X', 'X', 'O'])
mcts_100_vs_mcts_10000 (0, 10, ['O', 'O', 'O', 'O', 'O', 'X', 'X', 'X', 'X', 'X'])
mcts_1000_vs_mcts_10000 (2, 8, ['O', 'O', 'O', 'X', 'O', 'X', 'X', 'X', 'O', 'X'])
random wins: 0
mcts_100 wins: 0
mcts_1000 wins: 2
mcts_10000 wins: 8


Comparação entre AIs de ID3 decision tree:

In [90]:
misto1_dataset=dataset_1000+dataset_10000
misto2_dataset=dataset_100+dataset_1000
misto3_dataset=dataset_100+dataset_10000
super_dataset=dataset_100+dataset_1000+dataset_10000
datasets={100:dataset_100, 1000:dataset_1000, 10000:dataset_10000}
depths=[10,20,40,None]
player_names=["random", "ID3_100", "ID3_1000", "ID3_10000","misto_1", "misto_2", "misto_3", "super"]
players={}
for p in player_names:
    if p=="random":
        players[p]=RandomPlayer()
    elif p=="super" or p.split("_")[0]=="misto":
        for d in depths:
            tree=ID3DecisionTree(max_depth=d)
            tree.fit(super_dataset,label_col="move")
            players[f"{p}_{d}"]=ID3DecisionTreePlayer(tree)
            print(f"{p}_{d}")
    else:
        for d in depths:
            tree=ID3DecisionTree(max_depth=d)
            iter=int(p.split("_")[1])
            tree.fit(datasets[iter],label_col="move")
            players[f"{p}_{d}"]=ID3DecisionTreePlayer(tree)
            print(f"{p}_{d}")
print("players created")
n_random_games=10
n_games=2
player_names=list(players.keys())
results={}
for p in player_names:
    results[f"{p} wins"]=0
for i,p1 in enumerate(player_names):
    for p2 in player_names[i+1:]:
        game_name=f"{p1}_vs_{p2}"
        if p1=="random":
            result=simulate_games(players[p1],players[p2],n_random_games)
        else:
            result=simulate_games(players[p1],players[p2],n_games)
        results[f"{p1} wins"]+=result[0]
        results[f"{p2} wins"]+=result[1]
        print(f"{game_name} {result}")
for p in results:
    print(f"{p}: {results[p]}")

ID3_100_10
ID3_100_20
ID3_100_40
ID3_100_None
ID3_1000_10
ID3_1000_20
ID3_1000_40
ID3_1000_None
ID3_10000_10
ID3_10000_20
ID3_10000_40
ID3_10000_None
misto_1_10
misto_1_20
misto_1_40
misto_1_None
misto_2_10
misto_2_20
misto_2_40
misto_2_None
misto_3_10
misto_3_20
misto_3_40
misto_3_None
super_10
super_20
super_40
super_None
players created
random_vs_ID3_100_10 (2, 8, ['O', 'O', 'X', 'O', 'O', 'X', 'O', 'X', 'X', 'X'])
random_vs_ID3_100_20 (3, 7, ['O', 'O', 'O', 'O', 'X', 'O', 'X', 'O', 'X', 'X'])
random_vs_ID3_100_40 (2, 8, ['O', 'O', 'O', 'X', 'O', 'X', 'O', 'X', 'X', 'X'])
random_vs_ID3_100_None (4, 6, ['O', 'O', 'O', 'X', 'X', 'X', 'X', 'X', 'O', 'O'])
random_vs_ID3_1000_10 (2, 8, ['O', 'O', 'O', 'X', 'O', 'X', 'X', 'X', 'X', 'O'])
random_vs_ID3_1000_20 (0, 10, ['O', 'O', 'O', 'O', 'O', 'X', 'X', 'X', 'X', 'X'])
random_vs_ID3_1000_40 (2, 8, ['X', 'O', 'O', 'X', 'O', 'X', 'X', 'X', 'X', 'X'])
random_vs_ID3_1000_None (2, 8, ['O', 'O', 'O', 'X', 'O', 'X', 'X', 'O', 'X', 'X'])
random_vs

## Conclusões esperadas

- Mais iterações MCTS tendem a produzir jogadas mais estáveis, mas aumentam o tempo de decisão.
- A árvore ID3 aprende uma aproximação da política produzida pelo MCTS.
- Para PopOut, limitar a profundidade da árvore pode melhorar a generalização e reduzir overfitting.
- No Iris, a discretização por quantis reduz o tamanho da árvore e evita ramos demasiado específicos.